In [ ]:
# load packages
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.outliers_influence as oi
import statsmodels.stats.api as sms
import xgboost as xgb                        # Data visualization library, extension of matplotlib  
import random                                           # Randomly generating numbers
import math      

from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix 
from statsmodels.compat import lzip
from statsmodels.graphics.gofplots import qqplot

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import scipy.stats as stats                             # Fit of distribution plot
from sklearn.model_selection import train_test_split    # Split train-/testset     
from sklearn.tree import DecisionTreeClassifier         # Modeling CART Decision Tree  
from sklearn import metrics                             # Performance metrics
from sklearn.metrics import accuracy_score              # Accuracy of a model
from sklearn.metrics import classification_report       # Performance report of a classification model
#import graphviz as gv                                   # Needed to graph Decision Tree
#import pydotplus                                        # Needed to graph Decision Tree
from IPython.display import Image                       # Needed to graph Decision Tree
#from sklearn.externals.six import StringIO              # Needed to graph Decision Tree
from sklearn.tree import export_graphviz                # Needed to graph Decision Tree

#    Define function for the MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true-y_pred)/y_true))*100

print('All packages loaded')

## Load Data

In [ ]:
# Read file
os.chdir("")

inputdata = pd.read_csv("ice_cream_shop_dataset.csv", delimiter=',')

inputdata_orig = inputdata.copy() #save an original copy of the file, so that the unaltered data is stored
inputdata.describe()      #look at distribution of variables


## Inspection of Data

In [ ]:
# Inspection of data
corrMat = inputdata.corr().round(2)
f, ax = plt.subplots(figsize=(12,9))
sns.heatmap(corrMat, square=True, annot= True)

## Splitting in train- and testset

In [ ]:
# Create a trainset and testset
trainset, testset = ...


## Applying linear regression - build the model

In [ ]:
# a) Build the model using the training sets
y_train = ...
X_train = ...
X_train_incl_constant = sm.add_constant(X_train) # in this package, we have to add a constant explicitly

model = sm.OLS(...)
results = model.fit()
print(results.summary())


# Checking the model

#### checking non-linearity: plot independent variable versus dependent variable

In [ ]:
plt.scatter(... ,...)
plt.show()


In [ ]:
### If evidence for non-linearity is found, add transformations of features if necessary
### Fit the model including your new transformed features
...

# Checking multicollinearity

In [ ]:
X_train = ...
X_train_incl_constant = sm.add_constant(X_train)

# Using Vif scores
vif = [[X_train_incl_constant.columns[i],oi.variance_inflation_factor(X_train_incl_constant.values,i)] for i in range(1,X_train_incl_constant.shape[1])]
vif

In [ ]:
# now calculate the vif using the full variables set
...

In [ ]:
### Checking heteroskedasticity by plotting squared residuals against dependent and independent variables
squaredResids = results.resid
plt.scatter(... , ...)
plt.show()


In [ ]:
### In case heteroskedasticity might be present, Breusch-pagan test might be done, so that heteroskedasticity may be formally tested. 
### If p-value <0.05, homoskedasticity is rejected and heteroskedasticity is assumed
### First output is the LM statistic, the second its p-value, you want to look at this one
### Third output is the F statistic, the fourth its p-value
het_breuschpagan(results.resid, X_train_incl_constant)

In [ ]:
### So now we use a robust regression with white standard errors. In code: cov_type='HC0'
model = sm.OLS(y_train, X_train_incl_constant)
results = model.fit(cov_type='HC0')
print(results.summary())

Finally, we compute predictions

In [ ]:
X_test = #final selection of your X variables
predict_linear = ...

In [ ]:
# And test using plots and 3 measures
plt.scatter(..., ...)
plt.show()
print("MSE:")
print(mean_squared_error(..., ...))
print("MAE:")
print(mean_absolute_error(..., ...))
print("MAPE:")
print(mean_absolute_percentage_error(..., ...))

# logit regression

In [ ]:
X_train = trainset[['atmospheric_pressure','temperature','windspeed']]
X_train_constant = sm.add_constant(X_train)
logit_model = sm.GLM(trainset['rainy_day'], X_train_constant, family=sm.families.Binomial())
result = logit_model.fit()
print(result.summary())

In [ ]:
## Standardize independent variables so that relative importance can be assessed
standardized_trainset = ...

In [ ]:
# Now create the model for the standardized dataset and look at the results
...

In [ ]:
#Compute predictions for the test set, using the non-scaled model
X_test = ...
proba_logit = ...

# Now using the cut off, create predictions for the test set
cut_off = ...
predict_logit = ...

# Print confusion matrix
tn,fp,fn,tp = confusion_matrix(...).ravel()
print("True negative:")
print(tn)
print("False postive:")
print(fp)
print("False negative:")
print(fn)
print("True postive:")
print(tp)

# Decision tree

In [ ]:
# Define X and y 
X_variables = [...
              ]
y_variable = ...

X_train = trainset.loc[:, X_variables]
y_train = trainset[y_variable]
X_test = testset.loc[:, X_variables]
y_test = testset[y_variable]

In [ ]:
# a) Set model parameters 
#    Note: if Min_bucket is too large, the tree might not branch, if too small, the tree might get to big to interpret
Min_num_splits = ...                            # Minimum number of items to split
Min_bucket     = ...  # Minimum number of items per bucket
Max_depth      = ...                             # Maximum depth of final tree (nr of levels)

In [ ]:
# b) Estimate the model 
mytree = DecisionTreeClassifier(max_depth=Max_depth
                                ,min_samples_split=Min_num_splits
                                ,min_samples_leaf=Min_bucket
                                ,criterion="gini"              
                                ,splitter="best"
                                ,random_state=random.seed()
                                )

mytree.fit(X_train, y_train)     # Fit the model over the train set

In [ ]:
# c) Create predictions for the test set
preds_proba = mytree.predict_proba(...)
preds = mytree.predict(...)   # Cut-off point equals 0.5 by deafult

#    Show the first 5 lines of the prediction probabilities and corresponding prediction
pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1).head()

In [ ]:
#    Calculate the optimal cut-off point given certain costs
cost_TP = 50
cost_TN = 0
cost_FP = 50
cost_FN = 100
total_cost = math.inf

for i in np.linspace(0,1,100,endpoint=False):
    y_pred = (preds_proba[:,1]>i).astype('int')
    results = metrics.confusion_matrix(y_pred,y_test)
    TN = results[0][0]
    FN = results[0][1]
    FP = results[1][0]
    TP = results[1][1]
    
    # Calculate cutoff-point
    cost = TN*cost_TN + TP*cost_TP + FP*cost_FP + FN*cost_FN
    total_cost = min(total_cost,cost)
    if(total_cost == cost):
        opt_cutoff = i
        
print('Optimal cut-off:', opt_cutoff)

#    Create predictions for the test set according to optimal cutoff point
preds = (preds_proba[:,1] > opt_cutoff).astype('int')

In [ ]:
# d) Evaluate results
#    i. Create confusion matrix
print(pd.crosstab(..., ...))

In [ ]:
#    ii. Create classification report
print(classification_report(..., ...))

In [ ]:
#    iii. Get the feature importances of the tree
importances = mytree.feature_importances_ 
std = np.std([mytree.feature_importances_], axis=0)
indices = np.argsort(importances)[::-1]

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    print("Feature %d (%s) %f" % (indices[f], X_variables[indices[f]], importances[indices[f]]))
    importances_features.append(X_variables[indices[f]])

plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()

In [ ]:
#    iv. Create ROC curve
fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:,1])

#     v. Calculate AUC
roc_auc_tree = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC Decision Tree CART (AUC = %0.2f)' % (roc_auc_tree))
plt.title('ROC curve')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc="lower right")
plt.show()

In [ ]:
#    Plot tpr vs 1-fpr
i = np.arange(len(tpr)) # index for df
roc = pd.DataFrame({'fpr' : pd.Series(fpr, index=i),'tpr' : pd.Series(tpr, index = i)
                    ,'1-fpr' : pd.Series(1-fpr, index = i)
                    ,'tf' : pd.Series(tpr - (1-fpr), index = i)
                    ,'thresholds' : pd.Series(t, index = i)})
print(roc.loc[(roc.tf-0).abs().argsort()[:1]])
fig, ax = plt.subplots()
plt.plot(roc['tpr'])
plt.plot(roc['1-fpr'], color = 'red')
plt.title('Receiver operating characteristic')
plt.ylabel('True Positive Rate')
plt.xlabel('1-False Positive Rate')
ax.set_xticklabels([])

# K-means

In this section we will run a k-means clustering algorithm on the dataset and inspect whether we find any meaningful clustering

In [ ]:
trainset, testset = train_test_split(inputdata, test_size=0.3, random_state=13)
print(inputdata.shape)  #print number of rows and columns in inputdata
print(trainset.shape) #print number of rows and columns in trainset   
print(testset.shape)  #print number of rows and columns in testset
print(trainset.columns)

In [ ]:
# Fit the kmeans with two clusters
kmeans = KMeans(n_clusters=..., random_state=0, n_init="auto").fit(trainset)

In [ ]:
# Show the predictions
trainset['group'] = kmeans.labels_
trainset

In [ ]:
# Summarize statistics for group 0
trainset[trainset['group']==0].describe()

In [ ]:
# Summarize statistics for group 1
trainset[trainset['group']==1].describe()

In [ ]:
# Plot the cluster centers
pd.DataFrame(kmeans.cluster_centers_,columns=inputdata.columns)

In [ ]:
# Predict on the testset
testset['group'] = kmeans.predict(...)
testset

In [ ]:
# Create an elbow plot
WCSS = []
K = range(1,10)
for k in K:
    kmeanModel = KMeans(n_clusters=..., random_state=0, n_init="auto")
    kmeanModel.fit(...)
    WCSS.append(kmeanModel.inertia_)
    
plt.figure(figsize=(16,8))
plt.plot(K, distortions, 'bx-')
plt.xlabel('k')
plt.ylabel('WCSS')
plt.title('The Elbow Method showing the optimal k')
plt.show()

In [ ]:
# Does this differ when we standardize
scale = StandardScaler()
scaled_data = pd.DataFrame(scale.fit_transform(inputdata), columns=inputdata.columns)
trainset, testset = train_test_split(scaled_data, test_size=0.3, random_state=13)

In [ ]:
# Fit the kmeans with two clusters
kmeans = KMeans(n_clusters=2, random_state=0, n_init="auto").fit(trainset)
trainset = pd.DataFrame(scale.inverse_transform(trainset), columns=inputdata.columns)
trainset['group'] = kmeans.labels_
trainset

In [ ]:
# Summarize statistics for the two groups
trainset[trainset['group']==...].